# YMF example: Poisson problem — updated for the `ymf` package

This notebook is the **updated** version of `YMF-original.ipynb`. The
original prototyped a bespoke inline strictyaml schema (`Model` /
`ModelDomain` / `Domain` / `Grid` / `Topology` / `Geometry` / `DataItem`)
directly mirroring XDMF concepts, plus hand-rolled YAML→XDMF XML-writing
code.

That schema predates (and differs from) the `ymf` package's actual
`Problem` / `strong_form` / `solution_paths` / `archive` schema — see
`docs/ymf-schema.md` and `docs/ymf-schema-v0.2-delta.md`. This notebook
keeps the same worked example (a 3D Poisson problem archived as a
time-collection of tetrahedral meshes in HDF5, per `docs/ymf-schema.md`
§5 "XDMF Preservation") but:

1. Uses `ymf.load_ymf`/`ymf.validate_ymf` (real package schema) instead
   of the notebook's private inline `YMF` strictyaml schema.
2. Uses `ymf.write_xdmf` (in `src/ymf/xdmf.py`) instead of the hand-rolled
   `ElementTree` code that used to live directly in the notebook.
3. Demonstrates a full problem-definition YMF document (physics + weak
   form + discretization) *and* an `Archive.Domain` block for the mesh
   time-series output, since those are two distinct schema sections in
   the current package (the original notebook conflated them into one
   ad hoc structure).

## 1. A full YMF problem document (v0.1-compatible schema)

In [1]:
from ymf import load_ymf, validate_ymf, normalize_unknowns

poisson_problem_yaml = r"""
Problem:
  name: "Poisson 3D (unit cube, tetrahedral mesh)"
  physical_model:
    provenance: llm_derived
    processes: [diffusion]
    assumptions: [steady, constant_properties]

  strong_form:
    provenance: llm_derived
    unknowns: [u]
    equation_formulation: "Poisson"
    strong_form_expression: |
      Δu = f
    domain: |
      Ω = unit cube [0,1]^3
    boundary_regions:
      - name: "all_boundaries"
        geometry: "∂Ω"
    coefficients:
      f: "1.0"

  weak_forms:
    - label: "global"
      provenance: llm_derived
      derivation: |
        Multiply by test function v ∈ H1_0(Ω), integrate by parts.
      solution_spaces:
        trial: "H1"
        test: "H1_0"
      bilinear: |
        a(u,v) = ∫Ω ∇u·∇v dx
      linear: |
        l(v) = ∫Ω f·v dx

solution_paths:
  analytical: []
  discretizations:
    - name: "P1_tetrahedral"
      provenance: proteus_derived
      from_weak_form: "global"
      finite_element:
        fields: {family: CG, order: 1}
      discretization_details: "P1 on tetrahedral mesh, 687737 elements"
      solver:
        type: linear
        linear_solver: petsc
        tolerance: 1e-10

archive:
  format: xdmf
  time_collection: append
  version: "ymf-0.2"
"""

poisson_problem = validate_ymf(poisson_problem_yaml)
print('Problem name:', poisson_problem['Problem']['name'])
print('Unknowns (normalized):',
      normalize_unknowns(poisson_problem['Problem']['strong_form']['unknowns']))

Problem name: Poisson 3D (unit cube, tetrahedral mesh)
Unknowns (normalized): [{'name': 'u', 'units': None, 'std_name': None}]


## 2. The mesh archive data (Domain / TimeCollection / Grid)

This is the part of the original notebook's `poisson_data` YAML that
described the actual mesh time-series (two time steps of a tetrahedral
mesh, backed by HDF5 datasets). In the current schema this corresponds
to `docs/ymf-schema.md` §5 ("XDMF Preservation") — an `Archive.Domain`
block that mirrors XDMF's own `Domain`/`Grid` structure directly, since
*XDMF is a consumer of the archive, not a schema dependency of YMF*.

We keep this as a plain Python dict (rather than trying to force it
through the `Problem`-oriented `YMF_SCHEMA`, which doesn't model mesh
archives — only physics/discretization) and feed it straight to
`ymf.write_xdmf`.

In [2]:
domain = {
    "TimeCollection": {
        "Name": "Mesh Spatial_Domain",
        "Data": [
            {
                "Time": 0.0,
                "Topology": {
                    "Type": "Tetrahedron",
                    "NumberOfElements": 687737,
                    "DataItem": {
                        "Format": "HDF",
                        "DataType": "Int",
                        "Dimensions": [687737, 4],
                        "Data": "poisson_3.h5:/elementsSpatial_Domain0",
                    },
                },
                "Geometry": {
                    "Type": "XYZ",
                    "DataItem": {
                        "Format": "HDF",
                        "DataType": "Float",
                        "Precision": 8,
                        "Dimensions": [118084, 3],
                        "Data": "poisson_3.h5:/nodesSpatial_Domain0",
                    },
                },
            },
            {
                "Time": 1.0,
                "Topology": {
                    "Type": "Tetrahedron",
                    "NumberOfElements": 687737,
                    "DataItem": {
                        "Format": "HDF",
                        "DataType": "Int",
                        "Dimensions": [687737, 4],
                        "Data": "poisson_3.h5:/elementsSpatial_Domain1",
                    },
                },
                "Geometry": {
                    "Type": "XYZ",
                    "DataItem": {
                        "Format": "HDF",
                        "DataType": "Float",
                        "Precision": 8,
                        "Dimensions": [118084, 3],
                        "Data": "poisson_3.h5:/nodesSpatial_Domain1",
                    },
                },
            },
        ],
    }
}

## 3. Convert to XDMF using `ymf.write_xdmf` (package code, not notebook code)

In [3]:
from ymf import write_xdmf

write_xdmf(domain, "poisson.xmf")

with open("poisson.xmf") as f:
    print(f.read())

<?xml version="1.0" ?>
<!DOCTYPE Xdmf SYSTEM "Xdmf.dtd" []>
<Xdmf Version="2.0" xmlns:xi="http://www.w3.org/2001/XInclude">
  <Domain>
    <Grid Name="Mesh Spatial_Domain" GridType="Collection" CollectionType="Temporal">
      <Grid GridType="Uniform">
        <Time Value="0.0" Name="0" />
        <Topology Type="Tetrahedron" NumberOfElements="687737">
          <DataItem Format="HDF" DataType="Int" Precision="4" Dimensions="687737 4">poisson_3.h5:/elementsSpatial_Domain0</DataItem>
        </Topology>
        <Geometry Type="XYZ">
          <DataItem Format="HDF" DataType="Float" Precision="8" Dimensions="118084 3">poisson_3.h5:/nodesSpatial_Domain0</DataItem>
        </Geometry>
      </Grid>
      <Grid GridType="Uniform">
        <Time Value="1.0" Name="1" />
        <Topology Type="Tetrahedron" NumberOfElements="687737">
          <DataItem Format="HDF" DataType="Int" Precision="4" Dimensions="687737 4">poisson_3.h5:/elementsSpatial_Domain1</DataItem>
        </Topology>
        <

## 4. What changed vs. the original notebook

| Original (`YMF-original.ipynb`) | Updated (this notebook) |
|---|---|
| Inline `Model`/`ModelDomain`/`Domain`/`Grid`/... strictyaml schema defined *in the notebook* | Real `ymf.YMF_SCHEMA` from `src/ymf/schema.py`, versioned and tested |
| One YAML blob conflating physics (`Model.Equation`) and mesh archive (`Domain.TimeCollection`) | Two clear sections: a `Problem`/`solution_paths` document for physics+discretization, and a plain `Archive.Domain` dict for the mesh time-series |
| Hand-rolled `ElementTree` XML-building code, `indentXML()`, pasted inline | `ymf.write_xdmf()` / `ymf.build_xdmf_tree()` — tested package functions |
| `dirty_load(..., allow_flow_style=True)` called directly on an ad hoc schema | Same `allow_flow_style=True` need, now handled once inside `ymf.load_ymf`/`ymf.validate_ymf` so every caller gets it for free |

The original notebook is preserved unmodified at
`examples/notebooks/YMF-original.ipynb` for reference.